# 02 — Regressi Tabular dengan scikit-learn (Notebook)

Tujuan:
- Memuat data tabular (pakai hasil labs/01 jika ada, atau data sintetis)
- Memisahkan fitur/label, train/test split
- Membangun Pipeline (preprocessing numerik+kategori + model)
- Melatih dan mengevaluasi (RMSE, R²)
- Menyimpan pipeline terlatih ke `labs/07-fastapi-ml/model.joblib` (untuk API)

Catatan:
- Jika `datasets/processed/housing_clean.csv` belum ada, notebook akan membuat data sintetis.
- Lihat juga penjelasan di docs/05-ml-dasar.md dan docs/06-ml-pipeline-sklearn.md


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 1) Muat data: pakai hasil labs/01 jika ada, jika tidak buat sintetis

In [ ]:
processed_path = Path("ai-engineering-roadmap/datasets/processed/housing_clean.csv")
if processed_path.exists():
    df = pd.read_csv(processed_path)
else:
    rng = np.random.default_rng(42)
    n = 1200
    city = rng.choice(["A","B","C"], size=n, p=[0.4,0.4,0.2])
    condition = rng.choice(["poor","fair","good"], size=n, p=[0.2,0.5,0.3])
    area = rng.normal(80, 30, size=n).clip(20, 300)
    rooms = rng.integers(1, 7, size=n)
    age = rng.integers(0, 50, size=n)
    base_price = 30000
    price = (
        base_price + area * 1500 + rooms * 10000 - age * 800
        + np.where(city == "A", 10000, np.where(city == "B", 5000, 0))
        + np.where(condition == "good", 15000, np.where(condition == "fair", 5000, 0))
        + rng.normal(0, 15000, size=n)
    )
    df = pd.DataFrame({
        "city": city,
        "condition": condition,
        "area": area,
        "rooms": rooms,
        "age": age,
        "price": price.clip(10000, None)
    })
df.head()

## 2) Siapkan fitur/label dan train/test split

In [ ]:
target = "price"
y = df[target]
X = df.drop(columns=[target])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

## 3) Bangun Pipeline (preprocessing + model)

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_tf = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_tf = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_cols),
        ("cat", categorical_tf, cat_cols),
    ]
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("reg", LinearRegression()),
])

pipe

## 4) Latih, evaluasi, dan simpan model.joblib untuk FastAPI

In [ ]:
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)
print(f"RMSE: {rmse:,.2f} | R^2: {r2:.4f}")

# Simpan model untuk dipakai di API
out_dir = Path("ai-engineering-roadmap/labs/07-fastapi-ml")
out_dir.mkdir(parents=True, exist_ok=True)
model_path = out_dir / "model.joblib"
joblib.dump(pipe, model_path)
model_path

Selesai. Anda bisa menjalankan layanan API:

- `python ai-engineering-roadmap/labs/07-fastapi-ml/main.py`
- Buka docs: http://127.0.0.1:8000/docs

Atau gunakan task runner:
- `./tasks.ps1 run:pipeline` (script lain, non-notebook)
- `./tasks.ps1 run:api`
